# Jets before and after hadronization

Compare anti-$k_t$ jets from the same JEWEL event at two stages. The full `jewel-run-events` record contains the exact string inputs sent to Pythia8 in `hadronization_input` and the returned event in `hadronized_particles`. The string inputs can include beam remnants and recoil partons. The earlier cascade record is available as `particles`.

Use a notebook kernel with cppyy and install `pip install 'hepyy[hepbundle,jewel-cpp]' hepyy-utils numpy matplotlib`. Build the `jewel-run-events` executable from the [C++ checkout](../README.md#build-and-test). Set `JEWEL_BUILD_DIR` if you built outside `build-pythia8`. The Python wheel supports the hadronized final-state interface. This example uses the executable to obtain the paired records.

In [ ]:
%matplotlib inline
from collections import Counter
import json
import math
import os
from pathlib import Path
import subprocess
import tempfile

import matplotlib.pyplot as plt
import numpy as np

from hepyy_utils.jewel.matching import cluster_final_jets

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
build = Path(os.environ.get("JEWEL_BUILD_DIR", root / "build-pythia8")).expanduser().resolve()
runner = build / "jewel-run-events"
if not runner.is_file():
    raise FileNotFoundError(f"Build jewel-run-events or set JEWEL_BUILD_DIR (checked {runner}).")

MODES = ("vacuum", "medium")
ATTEMPTS = 40
SEED = 2601017
RADIUS = 0.4
PT_MIN = 25.0
MAX_ABS_ETA = 2.0
MATCH_DR = 0.3

## Generate one record per event

The medium run uses the 0–10% optical centrality setting, retains recoils, and requests the JEWEL thermal subtraction. `hadronized_particles` still contains the raw hadron record. `subtracted_particles` contains the corrected hadron momenta after subtraction using the incoming thermal references associated with retained recoils. The analysis removes entries flagged as thermal residuals and excludes neutrinos from all jet inputs. Jet-correlated recoil contributions can remain after subtraction.

The runner's validated subtraction is applied after hadronization. The vacuum raw-to-raw comparison isolates the fragmentation and decay stage for these events. In medium, raw partons vs raw hadrons keeps the same thermal treatment on both sides. The corrected-hadron result is shown alongside it to illustrate the separate effect of subtraction.

In [ ]:
def jet_axes(collection):
    momenta = np.asarray([p["momentum"] for p in collection
                          if abs(p["pdg"]) not in (12, 14, 16)], dtype=float).reshape(-1, 4)
    return cluster_final_jets(*momenta.T, radius=RADIUS)


def axis_distance(a, b):
    return math.hypot(a["eta"] - b["eta"], math.remainder(a["phi"] - b["phi"], math.tau))


def accepted(jet):
    return jet["pt"] >= PT_MIN and abs(jet["eta"]) < MAX_ABS_ETA


def analyze_event(event, mode, counts, samples):
    collections = {
        "Raw hadrons": (p for p in event["hadronized_particles"] if p["final"]),
    }
    if mode == "medium":
        collections["Subtracted hadrons"] = (p for p in event["subtracted_particles"]
                                             if p["final"] and not p["thermal_residual"])
    parton_jets = jet_axes(event["hadronization_input"])
    if not parton_jets:
        return
    # Choose the hardest parton-level anchor in full phase space. Search each
    # complete hadron jet list before applying acceptance to either jet.
    anchor = parton_jets[0]
    for label, particles in collections.items():
        jets = jet_axes(particles)
        closest = min(jets, key=lambda jet: axis_distance(anchor, jet), default=None)
        if not accepted(anchor):
            continue
        counts["leading parton selection", label] += 1
        if closest is None or axis_distance(anchor, closest) > MATCH_DR:
            counts["unmatched", label] += 1
        elif not accepted(closest):
            counts["matched outside jet cuts", label] += 1
        else:
            samples[label].append((anchor["pt"], closest["pt"]))


results = {}
for mode in MODES:
    labels = ["Raw hadrons"] + (["Subtracted hadrons"] if mode == "medium" else [])
    samples = {label: [] for label in labels}
    counts = Counter()
    with tempfile.TemporaryDirectory(prefix="jewel-before-after-") as work:
        path = Path(work) / "events.jsonl"
        args = [str(runner), str(path), str(ATTEMPTS), str(SEED), mode,
                "--ecm", "5020", "--pt-min", "40", "--pt-max", "100", "--isr-only",
                "--table-energy", "jewel", "--longitudinal-boost", "on",
                "--jewel-fragmentation", "--keep-recoils", "on"]
        if mode == "medium":
            args += ["--centrality", "0", "10", "--subtraction-radius", str(RADIUS)]
        run = subprocess.run(args, cwd=work, text=True, capture_output=True)
        if run.returncode:
            raise RuntimeError(run.stdout[-2000:] + run.stderr[-2000:])
        with path.open() as stream:
            header = json.loads(next(stream))
            assert header["schema"] == "jewel-hadron-development-v1"
            if mode == "medium":
                assert header["subtraction"]["radius"] == RADIUS
            for line in stream:
                event = json.loads(line)
                if event["type"] == "event":
                    analyze_event(event, mode, counts, samples)
                elif event["type"] == "summary":
                    summary = event

    results[mode] = {label: np.asarray(sample).reshape(-1, 2)
                     for label, sample in samples.items()}
    print(f'{mode.capitalize()} — {summary["accepted"]}/{ATTEMPTS} accepted events')
    for label in labels:
        print(f'  {label}: {len(samples[label])}/{counts["leading parton selection", label]} selected pairs, '
              f'{counts["unmatched", label]} unmatched, '
              f'{counts["matched outside jet cuts", label]} matched outside jet cuts')

## Check the paired response

Each point is the leading string-input jet and its nearest jet in the specified final collection in the same event. Matching uses $Delta Rleq0.3$ in pseudorapidity and azimuth. The $p_Tgeq25$ GeV and $|eta|<2$ cuts follow matching. The medium series can contain different passing pairs because correction can change a jet axis or move it across a cut. This is a small demonstration sample.

In [ ]:
colors = {"Raw hadrons": "#086f8f", "Subtracted hadrons": "#ae531d"}
fig, panels = plt.subplots(1, len(MODES), figsize=(6.8 * len(MODES), 5.1), squeeze=False,
                           layout="constrained")
fig.suptitle("Jet response on the same JEWEL events", fontsize=17, fontweight="bold")
for mode, ax in zip(MODES, panels[0]):
    all_pt = [value.max() for value in results[mode].values() if value.size]
    upper = max(max(all_pt, default=PT_MIN + 1) * 1.06, PT_MIN + 1)
    ax.plot([PT_MIN, upper], [PT_MIN, upper], color="#708091", lw=1, ls="--", label="Equal $p_T$")
    for label, pairs in results[mode].items():
        ax.scatter(*pairs.T, s=37, color=colors[label], alpha=.76,
                   edgecolor="white", linewidth=.35, label=f"{label} ({len(pairs)})")
    ax.set(title="Vacuum" if mode == "vacuum" else "0–10% Pb–Pb medium",
           xlabel="String-input jet $p_T$ [GeV]", ylabel="Matched hadron jet $p_T$ [GeV]",
           xlim=(PT_MIN, upper), ylim=(PT_MIN, upper), aspect="equal")
    ax.grid(alpha=.17)
    ax.legend(frameon=False, fontsize=9)
plt.show()